# Statistical analysis: the 10 required questions

Played matches only; scheduled fixtures are 'Coming soon' and never analysed as results. Source: PostgreSQL warehouse (`docker compose up -d`) and saved reports.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)


Results are produced by `fbplatform.analysis.stats` (assumption checks first, Holm adjustment inside each question). This notebook reads the saved results.

In [2]:
r = pd.read_csv(ROOT / 'reports' / 'statistics' / 'results.csv')
r['ci'] = [('' if pd.isna(a) else f'[{a:.3g}, {b:.3g}]') for a, b in zip(r.ci_low, r.ci_high)]
r.groupby('q').size()

q
Q1      5
Q10     7
Q2      6
Q3      6
Q4      3
Q5      4
Q6     16
Q7      4
Q8      3
Q9     16
dtype: int64

In [3]:
view = ['analysis', 'method', 'n', 'estimate', 'ci', 'p_value', 'p_adj_holm', 'effect_size_name', 'effect_size']
for q in ['Q1', 'Q2']:
    print(r[r.q == q].question.iloc[0]); display(r[r.q == q][view].round(4))

Q1 Has the average number of goals per match changed over time?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
0,WC: goals per match (regulation) vs edition year,"Poisson GLM, rate ratio per decade",488,1.0932,"[1.02, 1.17]",0.0092,0.0367,RR-1 (relative change per decade),0.0932
1,EURO: goals per match (regulation) vs edition year,"Poisson GLM, rate ratio per decade",261,0.9418,"[0.856, 1.04]",0.2201,0.4613,RR-1 (relative change per decade),-0.0582
2,COPA: goals per match (regulation) vs edition year,"Poisson GLM, rate ratio per decade",224,0.9420,"[0.846, 1.05]",0.2761,0.4613,RR-1 (relative change per decade),-0.0580
3,UNL: goals per match (regulation) vs edition year,"Poisson GLM, rate ratio per decade",762,1.1314,"[0.955, 1.34]",0.1538,0.4613,RR-1 (relative change per decade),0.1314
4,All competitions pooled (competition fixed effects),quasi-Poisson GLM + competition fixed effects,1735,1.0283,"[0.981, 1.08]",0.2450,0.2450,RR-1,0.0283


Q2 Are knockout matches lower-scoring than group matches?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
5,"WC: knockout vs group/league, goals in regulation",Mann-Whitney U (two-sided); Welch t shown as sensitivity,488,-0.2623,"[-0.611, 0.0854]",0.0963,0.2739,Cliff's delta (Cohen's d=-0.15),-0.0972
6,"EURO: knockout vs group/league, goals in regulation",Mann-Whitney U (two-sided); Welch t shown as sensitivity,261,0.4051,"[-0.0488, 0.851]",0.0913,0.2739,Cliff's delta (Cohen's d=0.27),0.1428
7,"COPA: knockout vs group/league, goals in regulation",Mann-Whitney U (two-sided); Welch t shown as sensitivity,224,0.6477,"[0.142, 1.18]",0.0188,0.0938,Cliff's delta (Cohen's d=0.38),0.2178
8,"UNL: knockout vs group/league, goals in regulation",Mann-Whitney U (two-sided); Welch t shown as sensitivity,762,0.6163,"[0.0713, 1.2]",0.0339,0.1357,Cliff's delta (Cohen's d=0.38),0.1829
9,"ALL: knockout vs group/league, goals in regulation",Mann-Whitney U (two-sided); Welch t shown as sensitivity,1735,0.2017,"[-0.0188, 0.422]",0.1134,0.2739,Cliff's delta (Cohen's d=0.12),0.0586
10,"Pooled, competition fixed effects: knockout rate ratio",quasi-Poisson GLM,1735,1.0807,"[0.997, 1.17]",0.0606,0.0606,RR-1,0.0807


In [4]:
for q in ['Q3', 'Q4', 'Q5']:
    print(r[r.q == q].question.iloc[0]); display(r[r.q == q][view].round(4))

Q3 Do teams that score first win more frequently?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
11,WC: result of the team that scored first (W 311 / D 71 /...,exact binomial sign test: win vs loss among decisive res...,432,0.7199,"[0.675, 0.762]",0.0000,0.0000,win/(win+loss) share (0.5 = no edge),0.8615
12,EURO: result of the team that scored first (W 162 / D 46...,exact binomial sign test: win vs loss among decisive res...,241,0.6722,"[0.609, 0.731]",0.0000,0.0000,win/(win+loss) share (0.5 = no edge),0.8308
13,COPA: result of the team that scored first (W 158 / D 40...,exact binomial sign test: win vs loss among decisive res...,216,0.7315,"[0.667, 0.789]",0.0000,0.0000,win/(win+loss) share (0.5 = no edge),0.8977
14,UNL: result of the team that scored first (W 484 / D 114...,exact binomial sign test: win vs loss among decisive res...,671,0.7213,"[0.686, 0.755]",0.0000,0.0000,win/(win+loss) share (0.5 = no edge),0.8689
15,ALL: result of the team that scored first (W 1115 / D 27...,exact binomial sign test: win vs loss among decisive res...,1560,0.7147,"[0.692, 0.737]",0.0000,0.0000,win/(win+loss) share (0.5 = no edge),0.8650
16,Homogeneity: first-scorer result distribution across com...,chi-square test of independence,1560,4.6452,,0.5901,0.5901,Cramer's V,0.0386


Q4 Is there a meaningful relationship between goals scored and tournament success?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
17,Group-stage goals scored/match vs number of knockout mat...,Spearman rank correlation (Pearson shown),490,0.6119,"[0.548, 0.667]",0.0,0.0,Spearman rho^2,0.3744
18,Group-stage goals scored/match vs points per match in th...,Spearman rank correlation (Pearson shown),490,0.7239,"[0.677, 0.761]",0.0,0.0,Spearman rho^2,0.5240
19,Reaching the knockout stage ~ group-stage goals scored/m...,logistic regression (odds ratio per +1 goal/match),490,18.4002,"[10.8, 31.2]",0.0,0.0,AUC (goals scored alone),0.8567


Q5 Are stronger defensive teams more likely to progress?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
20,Group-stage goals conceded/match: progressed vs eliminated,Mann-Whitney U,490,-0.8716,"[-0.997, -0.751]",0.0,0.0,Cliff's delta,-0.6415
21,"Joint logistic model: effect of goals conceded/match, ad...",logistic regression (odds ratio per +1 goal/match),490,0.0378,"[0.0183, 0.0782]",0.0,0.0,standardised OR,0.0753
22,"Joint logistic model: effect of goals scored/match, adju...",logistic regression (odds ratio per +1 goal/match),490,59.6357,"[25.7, 139]",0.0,0.0,standardised OR,20.5790
23,Which predicts progress better? AUC defence (0.821) vs a...,AUC difference with bootstrap CI,490,-0.0360,"[-0.0845, 0.0178]",NaN,NaN,AUC difference,-0.0360


In [5]:
for q in ['Q6']:
    print(r[r.q == q].question.iloc[0]); display(r[r.q == q][view].round(4).head(8))

Q6 Do penalty shootouts favour particular teams?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
24,Does the team listed first win more often? (68 shoot-outs),exact binomial test vs 0.5,68,0.4559,"[0.335, 0.581]",0.5446,0.5952,deviation from 0.5,-0.0441
25,Does the team with the better record so far in that tour...,exact binomial test vs 0.5,59,0.5763,"[0.441, 0.704]",0.2976,0.5952,deviation from 0.5,0.0763
26,Argentina: shoot-out record (6/11),exact binomial test vs 0.5 (Holm across teams),11,0.5455,"[0.234, 0.833]",1.0000,1.0000,deviation from 0.5,0.0455
27,Brazil: shoot-out record (5/9),exact binomial test vs 0.5 (Holm across teams),9,0.5556,"[0.212, 0.863]",1.0000,1.0000,deviation from 0.5,0.0556
28,Chile: shoot-out record (3/4),exact binomial test vs 0.5 (Holm across teams),4,0.7500,"[0.194, 0.994]",0.6250,1.0000,deviation from 0.5,0.2500
29,Colombia: shoot-out record (2/7),exact binomial test vs 0.5 (Holm across teams),7,0.2857,"[0.0367, 0.71]",0.4531,1.0000,deviation from 0.5,-0.2143
30,Croatia: shoot-out record (4/6),exact binomial test vs 0.5 (Holm across teams),6,0.6667,"[0.223, 0.957]",0.6875,1.0000,deviation from 0.5,0.1667
31,England: shoot-out record (3/7),exact binomial test vs 0.5 (Holm across teams),7,0.4286,"[0.099, 0.816]",1.0000,1.0000,deviation from 0.5,-0.0714


In [6]:
for q in ['Q7', 'Q8', 'Q9', 'Q10']:
    print(r[r.q == q].question.iloc[0]); display(r[r.q == q][view].round(4).head(8))

Q7 Is performance consistent across tournament editions?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
40,WC: share of variance in edition performance explained b...,intraclass correlation ICC(1) from one-way random-effect...,194,0.3342,"[0.185, 0.452]",0.0007,0.0027,"ICC (0 = no consistency, 1 = fully consistent)",0.3342
41,EURO: share of variance in edition performance explained...,intraclass correlation ICC(1) from one-way random-effect...,116,0.1276,"[-0.0204, 0.257]",0.0500,0.0999,"ICC (0 = no consistency, 1 = fully consistent)",0.1276
42,COPA: share of variance in edition performance explained...,intraclass correlation ICC(1) from one-way random-effect...,107,0.4792,"[0.189, 0.67]",0.0007,0.0027,"ICC (0 = no consistency, 1 = fully consistent)",0.4792
43,UNL: share of variance in edition performance explained ...,intraclass correlation ICC(1) from one-way random-effect...,270,0.0749,"[-0.0448, 0.188]",0.0640,0.0999,"ICC (0 = no consistency, 1 = fully consistent)",0.0749


Q8 Does extra time significantly affect match outcomes?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
44,"Of 97 knockout matches that went beyond 90 minutes, how ...",exact binomial CI (descriptive),97,0.7010,"[0.6, 0.79]",NaN,NaN,share,0.7010
45,"Goals per minute in extra time vs regulation, same 53 ma...",exact conditional (binomial) test of equal Poisson rates,53,1.4250,"[0.942, 2.12]",0.0881,0.0881,rate ratio,1.4250
46,Is the team with the better record so far less likely to...,"Fisher exact test (2x2), Woolf CI for the odds ratio",276,0.6053,"[0.358, 1.02]",0.0777,0.0777,phi coefficient,0.1135


Q9 Are some competitions statistically more competitive than others?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
47,Absolute goal margin (regulation) differs between compet...,Kruskal-Wallis H test,1735,0.0012,,0.1645,0.1645,"epsilon-squared (0.01 small, 0.06 medium, 0.14 large)",0.0012
48,Pairwise (all matches): WC vs EURO mean margin,"Mann-Whitney U, Holm across the 6 pairs",749,0.0804,"[-0.0971, 0.249]",0.7760,1.0000,Cliff's delta,0.0121
49,Pairwise (all matches): WC vs COPA mean margin,"Mann-Whitney U, Holm across the 6 pairs",712,-0.2279,"[-0.445, -0.00999]",0.0461,0.2310,Cliff's delta,-0.0893
50,Pairwise (all matches): WC vs UNL mean margin,"Mann-Whitney U, Holm across the 6 pairs",1250,-0.0347,"[-0.174, 0.111]",0.6074,1.0000,Cliff's delta,-0.0165
51,Pairwise (all matches): EURO vs COPA mean margin,"Mann-Whitney U, Holm across the 6 pairs",485,-0.3083,"[-0.525, -0.0987]",0.0385,0.2310,Cliff's delta,-0.1041
52,Pairwise (all matches): EURO vs UNL mean margin,"Mann-Whitney U, Holm across the 6 pairs",1023,-0.1151,"[-0.268, 0.0436]",0.4633,1.0000,Cliff's delta,-0.0291
53,Pairwise (all matches): COPA vs UNL mean margin,"Mann-Whitney U, Holm across the 6 pairs",986,0.1932,"[-0.00532, 0.403]",0.0821,0.3282,Cliff's delta,0.0732
54,Draw share differs between competitions (all matches),chi-square test of independence,1735,2.9021,,0.4070,0.4070,Cramer's V,0.0409


Q10 Are goal-scoring patterns significantly different between tournament formats?


,analysis,method,n,estimate,ci,p_value,p_adj_holm,effect_size_name,effect_size
63,"WC: goals per match by format (WC-32: 384 matches, mean ...",Mann-Whitney U,488,-0.4315,"[-0.812, -0.0393]",0.0294,0.0881,Cliff's delta,-0.1367
64,"WC: distribution of goals per match (0,1,2,3,4+) differs...",chi-square test of independence,488,5.8988,,0.2068,0.6205,Cramer's V,0.1099
65,"EURO: goals per match by format (EURO-16: 113 matches, m...",Mann-Whitney U,261,0.2588,"[-0.0997, 0.633]",0.2425,0.4850,Cliff's delta,0.0825
66,"EURO: distribution of goals per match (0,1,2,3,4+) diffe...",chi-square test of independence,261,2.2480,,0.6902,1.0000,Cramer's V,0.0928
67,"COPA: goals per match by format (COPA-10: 25 matches, me...",Kruskal-Wallis H,224,NaN,,0.6472,0.6472,epsilon-squared,0.0000
68,"COPA: distribution of goals per match (0,1,2,3,4+) diffe...",chi-square test of independence,224,4.0519,,0.8524,1.0000,Cramer's V,0.0951
69,All formats compared on group/league matches (8 formats),Kruskal-Wallis H test,1436,0.0033,,0.1123,0.1123,epsilon-squared,0.0033


### Limitations per analysis

In [7]:
display(r[['q', 'analysis', 'limitations']].drop_duplicates('limitations').head(25))

,q,analysis,limitations
0,Q1,WC: goals per match (regulation) vs edition year,few editions (confounded with format/host changes); matc...
4,Q1,All competitions pooled (competition fixed effects),"pools different formats/eras; trend is an association, n..."
5,Q2,"WC: knockout vs group/league, goals in regulation",extra-time matches without a known 90-min score are excl...
10,Q2,"Pooled, competition fixed effects: knockout rate ratio",association only; competition effects absorbed by fixed ...
11,Q3,WC: result of the team that scored first (W 311 / D 71 /...,reverse causation possible (strong teams score first and...
16,Q3,Homogeneity: first-scorer result distribution across com...,descriptive consistency check
17,Q4,Group-stage goals scored/match vs number of knockout mat...,group goals only; Euro 24-team format lets third-placed ...
18,Q4,Group-stage goals scored/match vs points per match in th...,group goals only (avoids counting knockout goals that fo...
19,Q4,Reaching the knockout stage ~ group-stage goals scored/m...,team-editions from the same tournament are not independe...
20,Q5,Group-stage goals conceded/match: progressed vs eliminated,descriptive; group goals only; Euro/Copa formats differ ...


### Descriptive statistics

In [8]:
pd.read_csv(ROOT / 'reports' / 'statistics' / 'descriptives.csv').round(3)

,competition,variable,n,mean,median,mode,variance,sd,p05,p25,p75,p95,iqr,skew,excess_kurtosis,dispersion_index_var_over_mean
0,ALL,goals_total,1779,2.526,2.0,2.0,2.734,1.654,0.0,1.0,3.0,6.0,2.0,0.657,0.356,1.082
1,ALL,goals90,1735,2.530,2.0,2.0,2.719,1.649,0.0,1.0,3.0,6.0,2.0,0.661,0.356,1.075
2,ALL,margin,1779,1.370,1.0,1.0,1.582,1.258,0.0,1.0,2.0,4.0,1.0,1.319,2.420,1.154
3,COPA,goals_total,248,2.512,2.0,1.0,3.028,1.740,0.0,1.0,4.0,6.0,3.0,0.522,-0.269,1.205
4,COPA,goals90,224,2.616,2.0,1.0,2.919,1.709,0.0,1.0,4.0,6.0,3.0,0.539,-0.220,1.116
5,COPA,margin,248,1.440,1.0,1.0,1.956,1.398,0.0,0.0,2.0,4.0,2.0,1.224,1.399,1.359
6,EURO,goals_total,277,2.462,2.0,2.0,2.336,1.528,0.0,1.0,3.0,5.0,2.0,0.663,0.543,0.949
7,EURO,goals90,261,2.490,2.0,2.0,2.266,1.505,0.0,1.0,3.0,5.0,2.0,0.608,0.232,0.910
8,EURO,margin,277,1.242,1.0,1.0,1.148,1.071,0.0,0.0,2.0,3.0,2.0,0.942,0.831,0.924
9,UNL,goals_total,766,2.501,2.0,2.0,2.671,1.634,0.0,1.0,3.0,6.0,2.0,0.605,0.176,1.068
